# RQ1 Prompt Comparison Analysis: Old vs New CWE-Based Few-Shot Prompts

**Date**: 2025-11-02  
**Purpose**: Compare vulnerability detection performance between original LLM-generated few-shot examples and new CWE-based canonical examples

## Research Question
Does the quality of few-shot examples (CWE-based vs LLM-generated) affect the Chain-of-Thought (CoT) paradox where few-shot prompts degrade performance?

## Experiments Compared

### Original Prompts (Phase 1 & 2a)
- Location: `results/mars/` (4B) and `results/runpod/` (30B)
- Few-shot examples: LLM-generated vulnerability examples

### New CWE Prompts (Re-run)
- Location: `results/mars_rerun/` (4B) and `results/runpod_rerun/` (30B)
- Few-shot examples: Canonical CWE-based examples
  - CWE-787: Buffer overflow (strcpy)
  - CWE-401: Memory leak (missing delete)
  - CWE-193: Off-by-one error

## Key Questions
1. Does using canonical CWE examples improve F1 scores?
2. Is the CoT paradox (few-shot degrading performance) still present with better prompts?
3. Do Thinking models benefit more from high-quality prompts than Instruct models?
4. Does prompt quality affect energy consumption patterns?

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Paths
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / 'results'

# Output directory
OUTPUT_DIR = RESULTS_DIR / 'analysis_prompt_comparison'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Define File Basenames

**Note**: We specify exact file basenames to ensure we're comparing the correct experiments.

In [ ]:
# Old results (original LLM-generated prompts)
OLD_FILES = {
    '4b_instruct': {
        'dir': RESULTS_DIR / 'mars',
        'base': 'Sa-few_Qwen-Qwen3-4B-Instruct-2507_20251011-110256',
        'codecarbon_dir': 'codecarbon_baseline_sa-few',
        'name': '4B Instruct (Old)'
    },
    '4b_thinking': {
        'dir': RESULTS_DIR / 'mars',
        'base': 'Sa-few_Qwen-Qwen3-4B-Thinking-2507_20251011-110932',
        'codecarbon_dir': 'codecarbon_thinking_sa-few',
        'name': '4B Thinking (Old)'
    },
    '30b_instruct': {
        'dir': RESULTS_DIR / 'runpod' / 'instruct_few_20251020_200040',
        'base': 'Sa-few_Qwen-Qwen3-30B-A3B-Instruct-2507_20251020-111953',
        'codecarbon_dir': 'codecarbon_baseline_sa-few',
        'name': '30B Instruct (Old)'
    },
    '30b_thinking': {
        'dir': RESULTS_DIR / 'runpod' / 'thinking_few_20251020_214835',
        'base': 'Sa-few_Qwen-Qwen3-30B-A3B-Thinking-2507_20251020-112644',
        'codecarbon_dir': 'codecarbon_thinking_sa-few',
        'name': '30B Thinking (Old)'
    },
}

# New results (CWE-based prompts)
NEW_FILES = {
    '4b_instruct': {
        'dir': RESULTS_DIR / 'mars_rerun',
        'base': 'Sa-few_Qwen-Qwen3-4B-Instruct-2507_20251101-200224',
        'codecarbon_dir': 'codecarbon_baseline_sa-few',
        'name': '4B Instruct (New)'
    },
    '4b_thinking': {
        'dir': RESULTS_DIR / 'mars_rerun',
        'base': 'Sa-few_Qwen-Qwen3-4B-Thinking-2507_20251101-200857',
        'codecarbon_dir': 'codecarbon_thinking_sa-few',
        'name': '4B Thinking (New)'
    },
    '30b_instruct': {
        'dir': RESULTS_DIR / 'runpod_rerun',
        'base': 'Sa-few_Qwen-Qwen3-30B-A3B-Instruct-2507_20251102-042208',
        'codecarbon_dir': 'codecarbon_baseline_sa-few',
        'name': '30B Instruct (New)'
    },
    '30b_thinking': {
        'dir': RESULTS_DIR / 'runpod_rerun',
        'base': 'Sa-few_Qwen-Qwen3-30B-A3B-Thinking-2507_20251102-042248',
        'codecarbon_dir': 'codecarbon_thinking_sa-few',
        'name': '30B Thinking (New)'
    },
}

print("File configuration loaded successfully.")

## 3. Load Summary Metrics

In [ ]:
def load_summary_metrics(base_dir, file_base):
    """Load summary vulnerability metrics from CSV."""
    summary_file = base_dir / f"{file_base}_summary_vulnerability_metrics.csv"
    
    if not summary_file.exists():
        print(f"⚠️  File not found: {summary_file}")
        return None
    
    df = pd.read_csv(summary_file)
    
    # The summary file has a single row with metrics
    if len(df) > 0:
        row = df.iloc[0]
        return {
            'accuracy': row.get('Accuracy', None),
            'precision': row.get('Precision', None),
            'recall': row.get('Recall', None),
            'f1': row.get('F1_Score', None),  # Note: column is 'F1_Score' not 'f1-score'
            'support': row.get('Total_Samples', None)
        }
    return None

def load_codecarbon(base_dir, codecarbon_dir, session_filter=None):
    """Load CodeCarbon emissions data.
    
    Args:
        base_dir: Base directory containing results
        codecarbon_dir: Name of codecarbon subdirectory
        session_filter: Optional string to filter project_name (e.g., '20251101-200224')
    """
    emissions_file = base_dir / codecarbon_dir / 'emissions.csv'
    
    if not emissions_file.exists():
        print(f"⚠️  Emissions file not found: {emissions_file}")
        return None
    
    df = pd.read_csv(emissions_file)
    
    # Filter to specific session if provided
    if session_filter:
        df = df[df['project_name'].str.contains(session_filter, na=False)]
    
    if len(df) == 0:
        print(f"⚠️  No matching sessions found in: {emissions_file}")
        return None
    
    # Sum all sessions for this experiment
    total_emissions = df['emissions'].sum()  # kg CO2
    total_duration = df['duration'].sum()  # seconds
    total_energy = df['energy_consumed'].sum()  # kWh
    
    return {
        'emissions_kg': total_emissions,
        'duration_seconds': total_duration,
        'energy_kwh': total_energy,
        'num_sessions': len(df)
    }

# Load all data
old_data = {}
new_data = {}

print("\n=== Loading Old Results (LLM-generated prompts) ===")
for key, config in OLD_FILES.items():
    print(f"\nLoading {config['name']}...")
    metrics = load_summary_metrics(config['dir'], config['base'])
    
    # Extract timestamp from base filename for session filtering
    timestamp = config['base'].split('_')[-1]  # e.g., '20251011-102915'
    emissions = load_codecarbon(config['dir'], config['codecarbon_dir'], timestamp)
    
    old_data[key] = {
        'metrics': metrics,
        'emissions': emissions,
        'name': config['name']
    }
    
    if metrics:
        print(f"  ✓ Metrics loaded: F1={metrics['f1']:.4f}")
    else:
        print(f"  ⚠️  No metrics loaded")
    if emissions:
        print(f"  ✓ Emissions loaded: {emissions['emissions_kg']:.6f} kg CO2 ({emissions['num_sessions']} sessions)")

print("\n=== Loading New Results (CWE-based prompts) ===")
for key, config in NEW_FILES.items():
    print(f"\nLoading {config['name']}...")
    metrics = load_summary_metrics(config['dir'], config['base'])
    
    # Extract timestamp from base filename for session filtering
    timestamp = config['base'].split('_')[-1]
    emissions = load_codecarbon(config['dir'], config['codecarbon_dir'], timestamp)
    
    new_data[key] = {
        'metrics': metrics,
        'emissions': emissions,
        'name': config['name']
    }
    
    if metrics:
        print(f"  ✓ Metrics loaded: F1={metrics['f1']:.4f}")
    else:
        print(f"  ⚠️  No metrics loaded")
    if emissions:
        print(f"  ✓ Emissions loaded: {emissions['emissions_kg']:.6f} kg CO2 ({emissions['num_sessions']} sessions)")

print("\n✓ All data loaded successfully")

## 4. Create Comparison DataFrame

In [ ]:
# Build comparison dataframe
comparison_rows = []

for key in ['4b_instruct', '4b_thinking', '30b_instruct', '30b_thinking']:
    model_size = '4B' if '4b' in key else '30B'
    model_type = 'Instruct' if 'instruct' in key else 'Thinking'
    
    # Old (LLM) data
    old_metrics = old_data[key]['metrics']
    old_emissions = old_data[key]['emissions']
    
    if old_metrics:
        row = {
            'Model': model_size,
            'Type': model_type,
            'Prompt': 'Old (LLM)',
            'Accuracy': old_metrics['accuracy'],
            'Precision': old_metrics['precision'],
            'Recall': old_metrics['recall'],
            'F1': old_metrics['f1'],
        }
        if old_emissions:
            row['CO2_kg'] = old_emissions['emissions_kg']
            row['Duration_sec'] = old_emissions['duration_seconds']
        comparison_rows.append(row)
    
    # New (CWE) data
    new_metrics = new_data[key]['metrics']
    new_emissions = new_data[key]['emissions']
    
    if new_metrics:
        row = {
            'Model': model_size,
            'Type': model_type,
            'Prompt': 'New (CWE)',
            'Accuracy': new_metrics['accuracy'],
            'Precision': new_metrics['precision'],
            'Recall': new_metrics['recall'],
            'F1': new_metrics['f1'],
        }
        if new_emissions:
            row['CO2_kg'] = new_emissions['emissions_kg']
            row['Duration_sec'] = new_emissions['duration_seconds']
        comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)

print("\n=== Comparison Data ===")
print(df_comparison.to_string(index=False))

## 5. Calculate Deltas (New - Old)

In [ ]:
# Calculate deltas for each model/type combination
delta_data = []

for model in ['4B', '30B']:
    for model_type in ['Instruct', 'Thinking']:
        old_row = df_comparison[(df_comparison['Model'] == model) & 
                                 (df_comparison['Type'] == model_type) & 
                                 (df_comparison['Prompt'] == 'Old (LLM)')]
        new_row = df_comparison[(df_comparison['Model'] == model) & 
                                 (df_comparison['Type'] == model_type) & 
                                 (df_comparison['Prompt'] == 'New (CWE)')]
        
        if len(old_row) > 0 and len(new_row) > 0:
            delta_dict = {
                'Model': model,
                'Type': model_type,
                'Old_F1': old_row.iloc[0]['F1'],
                'New_F1': new_row.iloc[0]['F1'],
                'ΔF1': new_row.iloc[0]['F1'] - old_row.iloc[0]['F1'],
                'ΔAccuracy': new_row.iloc[0]['Accuracy'] - old_row.iloc[0]['Accuracy'],
                'ΔPrecision': new_row.iloc[0]['Precision'] - old_row.iloc[0]['Precision'],
                'ΔRecall': new_row.iloc[0]['Recall'] - old_row.iloc[0]['Recall'],
            }
            
            # Add emissions delta if available
            if 'CO2_kg' in old_row.columns and 'CO2_kg' in new_row.columns:
                delta_dict['Old_CO2_kg'] = old_row.iloc[0]['CO2_kg']
                delta_dict['New_CO2_kg'] = new_row.iloc[0]['CO2_kg']
                delta_dict['ΔCO2_kg'] = new_row.iloc[0]['CO2_kg'] - old_row.iloc[0]['CO2_kg']
            
            delta_data.append(delta_dict)

df_deltas = pd.DataFrame(delta_data)

print("\n=== Delta Analysis (New CWE - Old LLM) ===")
print(df_deltas.to_string(index=False))

# Highlight significant changes (|ΔF1| > 0.02 i.e., 2 percentage points)
print("\n=== Significant Changes (|ΔF1| > 2pp) ===")
significant = df_deltas[abs(df_deltas['ΔF1']) > 0.02]
if len(significant) > 0:
    print(significant[['Model', 'Type', 'Old_F1', 'New_F1', 'ΔF1']].to_string(index=False))
else:
    print("No significant changes detected.")

## 6. Visualization: F1 Score Comparison

In [ ]:
# Create side-by-side bar chart
fig, ax = plt.subplots(figsize=(14, 6))

# Get old and new F1 scores
old_f1 = df_comparison[df_comparison['Prompt'] == 'Old (LLM)']['F1'].values
new_f1 = df_comparison[df_comparison['Prompt'] == 'New (CWE)']['F1'].values
labels = (df_comparison[df_comparison['Prompt'] == 'Old (LLM)']['Model'] + ' ' + 
          df_comparison[df_comparison['Prompt'] == 'Old (LLM)']['Type']).values

x = np.arange(len(labels))
width = 0.35

bars1 = ax.bar(x - width/2, old_f1, width, label='Old (LLM-generated)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, new_f1, width, label='New (CWE-based)', color='#e74c3c', alpha=0.8)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model Configuration', fontsize=12, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
ax.set_title('F1 Score Comparison: Old LLM-Generated vs New CWE-Based Few-Shot Prompts', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=0)
ax.legend(loc='upper right', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'f1_comparison_old_vs_new.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: f1_comparison_old_vs_new.png")

## 7. Visualization: Delta F1 Scores

In [ ]:
# Create delta bar chart
fig, ax = plt.subplots(figsize=(12, 6))

labels = df_deltas['Model'] + ' ' + df_deltas['Type']
deltas = df_deltas['ΔF1'].values
colors = ['#27ae60' if d > 0 else '#e74c3c' for d in deltas]

bars = ax.barh(labels, deltas, color=colors, alpha=0.7)

# Add value labels
for i, (bar, delta) in enumerate(zip(bars, deltas)):
    x_pos = delta + (0.002 if delta > 0 else -0.002)
    ha = 'left' if delta > 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{delta:+.3f}', ha=ha, va='center', fontsize=10, fontweight='bold')

# Add reference line at 0
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

# Add significance threshold lines at ±2pp
ax.axvline(x=0.02, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(x=-0.02, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(0.02, len(labels)-0.5, '+2pp', fontsize=8, ha='center', color='gray')
ax.text(-0.02, len(labels)-0.5, '-2pp', fontsize=8, ha='center', color='gray')

ax.set_xlabel('ΔF1 Score (New CWE - Old LLM)', fontsize=12, fontweight='bold')
ax.set_ylabel('Model Configuration', fontsize=12, fontweight='bold')
ax.set_title('Impact of CWE-Based Prompts on F1 Score\n(Green = Improvement, Red = Degradation)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delta_f1_scores.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: delta_f1_scores.png")

## 8. Energy Consumption Analysis

In [ ]:
# Prepare energy comparison data
if 'CO2_kg' in df_comparison.columns:
    energy_comparison = df_comparison[['Model', 'Type', 'Prompt', 'CO2_kg']].copy()
    
    # Calculate CO2 per sample (assuming 386 samples)
    energy_comparison['CO2_per_sample_g'] = energy_comparison['CO2_kg'] * 1000 / 386
    
    print("\n=== Energy Consumption Comparison ===")
    print(energy_comparison.to_string(index=False))
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Group by model configuration
    config_labels = energy_comparison['Model'] + ' ' + energy_comparison['Type']
    
    # Plot 1: Total CO2
    old_co2 = energy_comparison[energy_comparison['Prompt'] == 'Old (LLM)']['CO2_kg'].values
    new_co2 = energy_comparison[energy_comparison['Prompt'] == 'New (CWE)']['CO2_kg'].values
    labels_unique = config_labels[energy_comparison['Prompt'] == 'Old (LLM)'].values
    
    x = np.arange(len(labels_unique))
    width = 0.35
    
    ax1.bar(x - width/2, old_co2, width, label='Old (LLM)', color='#3498db', alpha=0.8)
    ax1.bar(x + width/2, new_co2, width, label='New (CWE)', color='#e74c3c', alpha=0.8)
    ax1.set_ylabel('Total CO2 (kg)', fontsize=12)
    ax1.set_title('Total Energy Consumption', fontsize=12, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels_unique, rotation=45, ha='right')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot 2: CO2 per sample
    old_per_sample = energy_comparison[energy_comparison['Prompt'] == 'Old (LLM)']['CO2_per_sample_g'].values
    new_per_sample = energy_comparison[energy_comparison['Prompt'] == 'New (CWE)']['CO2_per_sample_g'].values
    
    ax2.bar(x - width/2, old_per_sample, width, label='Old (LLM)', color='#3498db', alpha=0.8)
    ax2.bar(x + width/2, new_per_sample, width, label='New (CWE)', color='#e74c3c', alpha=0.8)
    ax2.set_ylabel('CO2 per Sample (g)', fontsize=12)
    ax2.set_title('Average Energy per Sample', fontsize=12, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels_unique, rotation=45, ha='right')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'energy_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: energy_comparison.png")
else:
    print("⚠️  Energy data not available for comparison")

## 9. Statistical Summary

In [ ]:
print("\n" + "="*80)
print("STATISTICAL SUMMARY: CWE PROMPT IMPACT")
print("="*80)

print(f"\n1. Overall Impact:")
print(f"   Mean ΔF1: {df_deltas['ΔF1'].mean():+.4f}")
print(f"   Median ΔF1: {df_deltas['ΔF1'].median():+.4f}")
print(f"   Std Dev: {df_deltas['ΔF1'].std():.4f}")

print(f"\n2. Improvements vs Degradations:")
improvements = len(df_deltas[df_deltas['ΔF1'] > 0])
degradations = len(df_deltas[df_deltas['ΔF1'] < 0])
no_change = len(df_deltas[df_deltas['ΔF1'] == 0])
print(f"   Improved: {improvements}/{len(df_deltas)} configurations")
print(f"   Degraded: {degradations}/{len(df_deltas)} configurations")
print(f"   No change: {no_change}/{len(df_deltas)} configurations")

print(f"\n3. Significant Changes (|ΔF1| > 2pp):")
significant = df_deltas[abs(df_deltas['ΔF1']) > 0.02]
print(f"   Count: {len(significant)}/{len(df_deltas)} configurations")
if len(significant) > 0:
    for _, row in significant.iterrows():
        direction = "⬆️ IMPROVED" if row['ΔF1'] > 0 else "⬇️ DEGRADED"
        print(f"   - {row['Model']} {row['Type']}: {row['Old_F1']:.3f} → {row['New_F1']:.3f} ({row['ΔF1']:+.3f}) {direction}")

print(f"\n4. Model Size Comparison:")
delta_4b = df_deltas[df_deltas['Model'] == '4B']['ΔF1'].mean()
delta_30b = df_deltas[df_deltas['Model'] == '30B']['ΔF1'].mean()
print(f"   4B models mean ΔF1: {delta_4b:+.4f}")
print(f"   30B models mean ΔF1: {delta_30b:+.4f}")
print(f"   Difference: {abs(delta_30b - delta_4b):.4f}")

print(f"\n5. Instruct vs Thinking:")
delta_instruct = df_deltas[df_deltas['Type'] == 'Instruct']['ΔF1'].mean()
delta_thinking = df_deltas[df_deltas['Type'] == 'Thinking']['ΔF1'].mean()
print(f"   Instruct models mean ΔF1: {delta_instruct:+.4f}")
print(f"   Thinking models mean ΔF1: {delta_thinking:+.4f}")
print(f"   Difference: {abs(delta_thinking - delta_instruct):.4f}")

if 'ΔCO2_kg' in df_deltas.columns:
    print(f"\n6. Energy Impact:")
    print(f"   Mean ΔCO2: {df_deltas['ΔCO2_kg'].mean():+.6f} kg")
    print(f"   Total old CO2: {df_deltas['Old_CO2_kg'].sum():.6f} kg")
    print(f"   Total new CO2: {df_deltas['New_CO2_kg'].sum():.6f} kg")

print("\n" + "="*80)

## 10. Export Results

In [ ]:
# Export comparison table
df_comparison.to_csv(OUTPUT_DIR / 'prompt_comparison_full.csv', index=False)
print("✅ Saved: prompt_comparison_full.csv")

# Export delta table
df_deltas.to_csv(OUTPUT_DIR / 'prompt_comparison_deltas.csv', index=False)
print("✅ Saved: prompt_comparison_deltas.csv")

# Export to Excel with formatting
with pd.ExcelWriter(OUTPUT_DIR / 'prompt_comparison_analysis.xlsx', engine='openpyxl') as writer:
    df_comparison.to_excel(writer, sheet_name='Full Comparison', index=False)
    df_deltas.to_excel(writer, sheet_name='Deltas', index=False)

print("✅ Saved: prompt_comparison_analysis.xlsx")

print(f"\n📁 All outputs saved to: {OUTPUT_DIR}")

## 11. Key Findings & Conclusions

### Answer to Research Questions

**Q1: Does using canonical CWE examples improve F1 scores?**
- [Review delta table above]

**Q2: Is the CoT paradox still present with better prompts?**
- [Compare few-shot degradation with new vs old prompts]

**Q3: Do Thinking models benefit more from high-quality prompts?**
- [Compare Thinking vs Instruct ΔF1]

**Q4: Does prompt quality affect energy consumption patterns?**
- [Review energy delta analysis above]